organize the code for analysis, order it according to tables and figures

In [ ]:
# import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from statannotations.Annotator import Annotator
from scipy import stats
from scipy.stats import ttest_ind, spearmanr, mannwhitneyu, kruskal

from plot_wrapper import EmptyPlot

import matplotlib.font_manager as fm


import matplotlib as mpl
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

mpl.rcParams.update({
    "font.family": "DejaVu Sans",
    "axes.unicode_minus": False,
})

import re

from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.utils import resample

from factor_analyzer import FactorAnalyzer

import statsmodels.api as sm
import scikit_posthocs as sp


In [ ]:
## read the converegent datasets
# import data
folderName = '../Data/'
fileName = 'in-person-assessment-dataset.csv'
df = pd.read_csv(folderName+fileName)

# rename the columns to keep it consistent with the df
df = df.rename(columns = {
    'SBP Drop' : 'dSBP', 
    'DBP Drop' : 'dDBP', 
    'ADFSCI Total Score': 'total_score',
    'Injury Level': 'Injury_level',
    'Time Since Injury (Years)': 'Injury_duration'
})



In [ ]:
# strategies for missing values in the above columns
def handle_missing_values(df, columns, strategy='median'):
        """
        Handle missing values in specified columns of a dataframe.
        Parameters:
            df (pd.DataFrame): The dataframe to process.
            columns (list): List of column names to process.
            strategy (str): 'median', 'mean', or 'none' for handling NaNs.

        Returns:
            pd.DataFrame: DataFrame with processed columns.
        """
        for col in columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            if strategy == 'median':
                median_value = df[col].median()
                df[col].fillna(median_value, inplace=True)
            elif strategy == 'mean':
                mean_value = df[col].mean()
                df[col].fillna(mean_value, inplace=True)
            # if strategy == 'none', do nothing
        return df

# Add descriptor for OH according to the definition
df['has_OH_SBP'] = (df.dSBP <= -20).astype('int') # though this column might already exist in the dataset

#calculate OH severity score (Q22 and Q24), all columns start with Q22_
oh_severity_cols = [col for col in df.columns if re.match(r'Q(22|24)', col)]


#calculate OH frequency score (Q17, Q19-21, Q23), all columns start with Q17_, Q19_, Q20_, Q21_, Q23_, Q24_
oh_frequency_cols = [col for col in df.columns if re.match(r'Q(17|19|20|21|23)', col)]


# also for different contexts
transfer_cols = [col for col in df.columns if col.startswith("Q19")] + \
                [col for col in df.columns if col.startswith("Q22_R") and "_C1" in col]
meal_cols = [col for col in df.columns if col.startswith("Q20")] + \
            [col for col in df.columns if col.startswith("Q22_R") and "_C2" in col]
exercise_cols = [col for col in df.columns if col.startswith("Q21")] + \
                [col for col in df.columns if col.startswith("Q22_R") and "_C3" in col]

#calculate AD score
ad_cols = [col for col in df.columns if re.match(r"Q(3[7-8]|4[0-8])(_.*)?$", col)]


# Example usage:
df = handle_missing_values(
    df,
    oh_severity_cols + oh_frequency_cols + transfer_cols + meal_cols + exercise_cols,
    strategy='None'  # or 'mean', or 'none'
)


df['OH_severity_score'] = df[oh_severity_cols].sum(axis=1)
df['OH_frequency_score'] = df[oh_frequency_cols].sum(axis=1)
df['OH_domain_score'] = df['OH_severity_score'] + df['OH_frequency_score']

# Compute combined symptom scores for each context
df['combined_transfer'] = df[transfer_cols].sum(axis=1)
df['combined_meal'] = df[meal_cols].sum(axis=1)
df['combined_exercise'] = df[exercise_cols].sum(axis=1)



In [ ]:
def detect_and_remove_outliers(df, score_cols=['OH_domain_score', 'AD_domain_score', 'total_score'], 
                                drop_outliers=True):
    """
    Detect outliers using IQR method and optionally remove them.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Input dataframe
    score_cols : list or str
        Column name(s) to check for outliers. Can be a single column name or list of column names.
        Default: ['OH_domain_score', 'AD_domain_score', 'total_score']
    drop_outliers : bool
        If True, remove outliers from dataframe. If False, only identify them.
        Default: True
    
    Returns:
    --------
    dict : Dictionary containing:
        - 'summary': DataFrame with IQR statistics for each column
        - 'outliers': DataFrame with all identified outlier rows
        - 'df_cleaned': DataFrame with outliers removed (if drop_outliers=True) or original df (if False)
    """
    # Convert single column name to list
    if isinstance(score_cols, str):
        score_cols = [score_cols]
    
    # Step 1: Identify outliers using IQR method
    outlier_summary = []
    for c in score_cols:
        Q1 = df[c].quantile(0.25)
        Q3 = df[c].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        outliers = df[(df[c] < lower) | (df[c] > upper)]

        outlier_summary.append({
            'Variable': c,
            'Q1': round(Q1, 2),
            'Q3': round(Q3, 2),
            'IQR': round(IQR, 2),
            'Lower Bound': round(lower, 2),
            'Upper Bound': round(upper, 2),
            'Outlier Count': len(outliers)
        })

    summary_df = pd.DataFrame(outlier_summary)

    # Step 2: Extract bounds
    outlier_bounds = summary_df.set_index('Variable')[['Lower Bound', 'Upper Bound']].to_dict(orient='index')

    # Step 3: Compare each row to bounds
    comparison_results = []
    outlier_indices = set()

    for idx, row in df.iterrows():
        for var in score_cols:
            value = row[var]
            lower = outlier_bounds[var]['Lower Bound']
            upper = outlier_bounds[var]['Upper Bound']
            is_outlier = value < lower or value > upper
            if is_outlier:
                outlier_indices.add(idx)
                comparison_results.append({
                    'Index': idx,
                    'Participant': row.get('Participant', idx),  # Use index if Participant column doesn't exist
                    'Variable': var,
                    'Below Lower Bound': value < lower,
                    'Exceeds Upper Bound': value > upper,
                    'Is Outlier': is_outlier
                })

    comparison_df = pd.DataFrame(comparison_results)

    # Step 4: Remove outliers if requested
    if drop_outliers:
        df_cleaned = df.drop(index=outlier_indices).reset_index(drop=True)
    else:
        df_cleaned = df.copy()

    # Return results
    return {
        'summary': summary_df,
        'outliers': comparison_df,
        'df_cleaned': df_cleaned,
        'n_outliers_removed': len(outlier_indices)
    }

outlier_results = detect_and_remove_outliers(df, score_cols=['OH_domain_score'], drop_outliers=False)
df = outlier_results['df_cleaned']
print("outliers", outlier_results['outliers'])
df.head(5)

In [ ]:
## Code for Table 1
# Table 1, basic stats
table1 = {
    "Sample Size (n)": len(df),
    "OH Prevalence (ΔSBP ≥ 20 mmHg)": f"{df['has_OH_SBP'].sum()} / {len(df)} ({round(df['has_OH_SBP'].sum() / len(df) * 100, 1)}%)",
    "Age (years)": f"{round(df['Age'].mean(), 1)} ± {round(df['Age'].std(), 1)}",
    "Sex (M/F)": f"{(df['Sex'] == 'M').sum()}/{(df['Sex'] == 'F').sum()}",
    "Injury Level (Cervical/Thoracic)": f"{(df['NLI'].str.startswith('C')).sum()}/{(df['NLI'].str.startswith('T')).sum()}",
    "AIS Grade": ', '.join([f"{grade}: {(df['AIS'] == grade).sum()}" for grade in sorted(df['AIS'].unique())]),
    "Time Since Injury (years)": f"{round(df['Injury_duration'].mean(), 1)} ± {round(df['Injury_duration'].std(), 1)}"
}

# Convert dictionary to DataFrame
table1_df = pd.DataFrame(list(table1.items()), columns=["Characteristic", "Value"])
print("Table 1:")
display(table1_df)

In [ ]:
# calculate the floor effects and ceiling effects for OH and AD subscales and total score
# Floor effect: percentage of participants scoring at the minimum
# Ceiling effect: percentage of participants scoring at the maximum
# COSMIN criterion: floor/ceiling effects <15% are acceptable

def calculate_floor_ceiling(df, column):
    """Calculate floor and ceiling effects for a given score column."""
    min_score = df[column].min()
    max_score = df[column].max()
    n = len(df[column].dropna())
    
    floor_n = (df[column] == min_score).sum()
    ceiling_n = (df[column] == max_score).sum()
    
    floor_pct = (floor_n / n) * 100
    ceiling_pct = (ceiling_n / n) * 100
    
    return {
        'min': min_score,
        'max': max_score,
        'floor_n': floor_n,
        'floor_pct': floor_pct,
        'ceiling_n': ceiling_n,
        'ceiling_pct': ceiling_pct,
        'interpretation': 'Acceptable' if floor_pct < 15 and ceiling_pct < 15 else 'Potentially problematic'
    }

print("Floor and Ceiling Effects Analysis:")
print("=" * 60)

for score_name, col_name in [('OH Domain Score', 'OH_domain_score')]:
    results = calculate_floor_ceiling(df, col_name)
    print(f"\n{score_name}:")
    print(f"  Range: {results['min']} – {results['max']}")
    print(f"  Floor effect: {results['floor_n']} participants ({results['floor_pct']:.1f}%)")
    print(f"  Ceiling effect: {results['ceiling_n']} participants ({results['ceiling_pct']:.1f}%)")
    print(f"  COSMIN criterion (<15%): {results['interpretation']}")

In [ ]:
# 2. Internal Consistency (McDonald's ω ≥ 0.70)

import factor_analyzer as fa
fa_model = fa.FactorAnalyzer(n_factors=1, rotation=None)

# Impute missing values instead of dropping all rows, and remove constant columns
# Separate the questions into OH and AD subscales for more detailed analysis
# AD: Q7-Q16 (Q8, Q10-Q16), excluding Q9 which is non-numeric
# OH: Q17-Q24, excluding Q18 which is non-numeric

df_q = df[[col for col in df.columns if col.startswith('Q')]]
# Compute McDonald's ω
def mcdonalds_omega(loadings, variances):
    sum_loadings = np.sum(loadings)
    sum_loadings_squared = sum_loadings ** 2
    sum_variances = np.sum(variances)
    omega = sum_loadings_squared / (sum_loadings_squared + sum_variances)
    return omega

# Create separate dataframes for AD and OH subscales
ad_cols = [col for col in df_q.columns if any(f'Q{i}_' in col or col == f'Q{i}' for i in range(7, 17)) and 'Q9' not in col]
oh_cols = [col for col in df_q.columns if any(f'Q{i}_' in col or col == f'Q{i}' for i in range(17, 25)) and 'Q18' not in col]

df_ad = df_q[ad_cols]
df_ad = df_ad.apply(pd.to_numeric, errors='coerce')
df_oh = df_q[oh_cols]
df_oh = df_oh.apply(pd.to_numeric, errors='coerce')
df_q = pd.concat([df_ad, df_oh], axis=1)

print(f"AD subscale questions: {len(ad_cols)} columns")
print(f"OH subscale questions: {len(oh_cols)} columns")

# calculate internal consistency for OH, AD, and total scale
for subscale_name, subscale_df in [('AD Subscale', df_ad), ('OH Subscale', df_oh), ('Total Scale', df_q)]:
    
    X = subscale_df.astype(float)
    X = X.fillna(X.median())
    X = X.loc[:, X.var() > 0]

    fa_model.fit(X)
    loadings = fa_model.loadings_

    # Compute McDonald's ω
    variances = fa_model.get_uniquenesses()
    omega = mcdonalds_omega(loadings.flatten(), variances)
    print(f"McDonald's ω for {subscale_name}: {omega:.2f}")


In [ ]:
# 3. Known-Groups Validity (Discriminative Validity)
def known_groups_validity(df, score_column, group_column='has_OH_SBP'):
    """
    Test known-groups (discriminative) validity.
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Input dataframe
    score_column : str
        Name of the column containing the scores to compare
    group_column : str
        Name of the column defining the groups (default: 'has_OH_SBP')
    
    Returns:
    --------
    dict : Dictionary containing t-statistic, p-value, Cohen's d, and group statistics
    """
    scores_with_condition = df.loc[df[group_column] == 1, score_column]
    scores_without_condition = df.loc[df[group_column] == 0, score_column]
    
    # Perform independent samples t-test
    t_stat, p_value = stats.ttest_ind(scores_with_condition, scores_without_condition)
    # do a whitney u test as well
    u_stat, p_value_u = mannwhitneyu(scores_with_condition, scores_without_condition)
    
    # Calculate Cohen's d effect size
    mean_with = scores_with_condition.mean()
    mean_without = scores_without_condition.mean()
    pooled_std = np.sqrt(((len(scores_with_condition) - 1) * scores_with_condition.std()**2 + 
                           (len(scores_without_condition) - 1) * scores_without_condition.std()**2) / 
                          (len(scores_with_condition) + len(scores_without_condition) - 2))
    cohens_d = (mean_with - mean_without) / pooled_std

    # estimate 95% confidence interval for Cohen's d
    se_d = np.sqrt((len(scores_with_condition) + len(scores_without_condition)) / (len(scores_with_condition) * len(scores_without_condition)) + 
                   (cohens_d**2) / (2 * (len(scores_with_condition) + len(scores_without_condition))))
    ci_lower = cohens_d - 1.96 * se_d
    ci_upper = cohens_d + 1.96 * se_d

    # Display results
    print(f"Known-Groups Validity Results:")
    print(f"=" * 50)
    print(f"With OH (n={len(scores_with_condition)}): Mean = {mean_with:.1f}, Std = {scores_with_condition.std():.1f}")
    print(f"Without OH (n={len(scores_without_condition)}): Mean = {mean_without:.1f}, Std = {scores_without_condition.std():.1f}")
    print(f"\nIndependent samples t-test:")
    print(f"t-statistic: {t_stat:.4f}")
    print(f"p-value: {p_value:.4f}")
    print(f"Mann-Whitney U test p-value: {p_value_u:.4f}")
    print(f"Cohen's d: {cohens_d:.4f} (95% CI: {ci_lower:.4f} to {ci_upper:.4f})")
    print(f"\nInterpretation:")
    if cohens_d >= 0.8:
        effect_size_interpretation = "large effect (d ≥ 0.8)"
    elif cohens_d >= 0.5:
        effect_size_interpretation = "moderate effect (d ≥ 0.5)"
    else:
        effect_size_interpretation = "small effect (d < 0.5)"
    print(f"Effect size: {effect_size_interpretation}")
    
    return {
        't_stat': t_stat,
        'p_value': p_value,
        'cohens_d': cohens_d,
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'mean_with': mean_with,
        'mean_without': mean_without,
        'n_with': len(scores_with_condition),
        'n_without': len(scores_without_condition)
    }

# Usage example:
print('OH domain results:')
results = known_groups_validity(df, 'OH_domain_score')
print(f"p-value: {results['p_value']:.3f}, Cohen's d: {results['cohens_d']:.2f} (95% CI: {results['ci_lower']:.2f} to {results['ci_upper']:.2f})")

print('\nAD domain results:')
results = known_groups_validity(df, 'AD_domain_score')
print(f"p-value: {results['p_value']:.3f}, Cohen's d: {results['cohens_d']:.2f} (95% CI: {results['ci_lower']:.2f} to {results['ci_upper']:.2f})")

print('\ntotal score results:')
results = known_groups_validity(df, 'total_score')
print(f"p-value: {results['p_value']:.3f}, Cohen's d: {results['cohens_d']:.2f} (95% CI: {results['ci_lower']:.2f} to {results['ci_upper']:.2f})")

In [ ]:
# 4. Convergent Validity
#Correlates ADFSCI hypotension with a gold-standard measure (e.g., tilt-table BP drop or another established questionnaire)
corr, p_value = stats.spearmanr(df['dSBP'], df['OH_domain_score'])
print(f"Convergent Validity Results:")
print(f"Spearman correlation coefficient (r): {corr:.4f}")
print(f"p-value: {p_value:.4f}")

In [ ]:
## Code for Table 2
# Table 2. Internal structure of hypotension symptom score

hss_col = 'OH_domain_score'

# Number of items in the PROM (Q17-Q24)
n_items = len([col for col in df.columns if col.startswith('Q') and 'Q17' <= col <= 'Q24']) - 1  # excluding Q18 which is non-numeric

# Floor and ceiling effects (reuses calculate_floor_ceiling from Cell 6)
fc_hss = calculate_floor_ceiling(df, hss_col)

# McDonald's ω for the full PROM (reuses df_q, fa_model, mcdonalds_omega from Cell 7)
X_total = df_q.astype(float).fillna(df_q.median())
X_total = X_total.loc[:, X_total.var() > 0]
fa_model.fit(X_total)
omega_total = mcdonalds_omega(fa_model.loadings_.flatten(), fa_model.get_uniquenesses())

table2 = {
    "Number of items in the PROM": n_items,
    "Mean HSS": f"{round(df[hss_col].mean(), 1)} ± {round(df[hss_col].std(), 1)}",
    "Range of HSS": f"{int(fc_hss['min'])} – {int(fc_hss['max'])}",
    "Floor count (% at minimum score)": f"{fc_hss['floor_n']} ({round(fc_hss['floor_pct'], 1)}%)",
    "Ceiling count (% at maximum score)": f"{fc_hss['ceiling_n']} ({round(fc_hss['ceiling_pct'], 1)}%)",
    "McDonald’s ω (internal consistency)": f"{round(omega_total, 2)}",
}

table2_df = pd.DataFrame(list(table2.items()), columns=["Property", "Value"])
print("\nTable 2:")
display(table2_df)

In [ ]:
## Code for Table 3
# Table 3. Summary of validity tests

import io, contextlib
from scipy.stats import mannwhitneyu, f_oneway
from statsmodels.stats.multicomp import pairwise_tukeyhsd

hss_col = 'OH_domain_score'
group_col = 'has_OH_SBP'

# Known-groups validity: Cohen's d and CI (suppress verbose print from known_groups_validity)
_buf = io.StringIO()
with contextlib.redirect_stdout(_buf):
    kgv = known_groups_validity(df, hss_col, group_column=group_col)

# Mann-Whitney U between OH+ and OH- (two-sided)
# Groups match Figure 2: dSBP <= -20 threshold, one-sided test
oh_pos = df[df['dSBP'] <= -20][hss_col].dropna()
oh_neg = df[df['dSBP'] > -20][hss_col].dropna()
_, p_mwu = mannwhitneyu(oh_pos, oh_neg, alternative='greater')

# Quartile analysis: split by OH domain score quartile, compare ΔSBP across groups
# Mirrors Figure 2 quartile analysis (ANOVA + Tukey HSD post-hoc)
df['hss_quartile'] = pd.qcut(df[hss_col], 4, labels=['Q1', 'Q2', 'Q3', 'Q4'])
_df_q = df.dropna(subset=[hss_col, 'dSBP']).copy()
q_dsbp = [_df_q[_df_q['hss_quartile'] == q]['dSBP'] for q in ['Q1', 'Q2', 'Q3', 'Q4']]
_, p_anova = f_oneway(*q_dsbp)
tukey = pairwise_tukeyhsd(endog=_df_q['dSBP'], groups=_df_q['hss_quartile'], alpha=0.05)
tukey_res = pd.DataFrame(data=tukey._results_table.data[1:], columns=tukey._results_table.data[0])
p_q1_q3 = float(tukey_res[(tukey_res['group1'] == 'Q1') & (tukey_res['group2'] == 'Q3')]['p-adj'].values[0])
p_q1_q4 = float(tukey_res[(tukey_res['group1'] == 'Q1') & (tukey_res['group2'] == 'Q4')]['p-adj'].values[0])

# Convergent validity: Spearman correlation with SBP and DBP drops
rho_sbp, p_sbp = stats.spearmanr(df['dSBP'], df[hss_col])
rho_dbp, p_dbp = stats.spearmanr(df['dDBP'], df[hss_col])

table3 = {
    "OH+ / OH−": f"{kgv['n_with']} / {kgv['n_without']}",
    "Mean HSS in OH+ group": f"{round(kgv['mean_with'], 1)} ± {round(oh_pos.std(), 1)}",
    "Mean HSS in OH− group": f"{round(kgv['mean_without'], 1)} ± {round(oh_neg.std(), 1)}",
    "Known-groups validity": f"Cohen's d = {round(kgv['cohens_d'], 2)} (95% CI: {round(kgv['ci_lower'], 2)}–{round(kgv['ci_upper'], 2)})",
    "Mann-Whitney U test between groups": f"P = {round(p_mwu, 3)}",
    "Quartile analysis": f"ANOVA (p={round(p_anova, 3)}); Q1 v. Q3 p = {round(p_q1_q3, 3)} and Q1 v. Q4 p = {round(p_q1_q4, 3)}",
    "Convergent validity (ΔSBP and HSS)": f"Spearman ρ = {round(rho_sbp, 2)} (p = {round(p_sbp, 3)})",
    "Convergent validity (ΔDBP and HSS)": f"Spearman ρ = {round(rho_dbp, 2)} (p = {round(p_dbp, 3)})",
}

table3_df = pd.DataFrame(list(table3.items()), columns=["Property", "Value"])
display(table3_df)

In [ ]:
# =========================
# Global style
# =========================
sns.set_style("white")
MAIN_COLOR = "#f08080"
MEDIAN_COLOR = "darkred"

# =========================
# Prepare data
# =========================
df['SBP_Drop_Group'] = df['has_OH_SBP'].replace({0: 'OH-', 1: 'OH+'})

# =========================
# Create EmptyPlot canvas
# =========================
plot = EmptyPlot(nrows=2, ncols=2, figsize=(16, 14))
fig, ax = plot.fig, plot.ax

# =========================
# Fig 2a — Group comparison
# =========================
# put OH- first
df['SBP_Drop_Group'] = pd.Categorical(
    df['SBP_Drop_Group'],
    categories=['OH-', 'OH+'],
    ordered=True
)
sns.boxplot(
    data=df,
    x='SBP_Drop_Group',
    y='OH_domain_score',
    color=MAIN_COLOR,
    saturation=1,
    medianprops=dict(linewidth=3, color=MEDIAN_COLOR),
    boxprops=dict(linewidth=0, alpha=0.7),
    showfliers=False,
    ax=ax[0, 0]
)

ax[0, 0].set_ylabel('ADFSCI OH Domain Score', fontsize=14)
ax[0, 0].set_xlabel('')
ax[0, 0].tick_params(axis='both', labelsize=12)
ax[0, 0].set_title('a', loc='left', fontsize=16, fontweight='bold')

# stats
group1 = df[df['has_OH_SBP'] == 1]['OH_domain_score']
group2 = df[df['has_OH_SBP'] == 0]['OH_domain_score']
# use mann whitney u test
u_stat, u_p = mannwhitneyu(group1, group2, alternative='greater')

ax[0, 0].text(
    0.5, 0.95,
    f"one-sided Mann-Whitney U test\np = {u_p:.3f}",
    ha='center', va='top',
    transform=ax[0, 0].transAxes,
    fontsize=11
)

# =========================
# Fig 2b — Quartiles vs ΔSBP
# =========================

import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

df['binned_ADFSCI_hypo'], bins = pd.qcut(
    df['OH_domain_score'],
    q=4,
    labels=[f'Q{i+1}' for i in range(4)],
    retbins=True
)

range_labels = [f"Q{i+1}\n({bins[i]:.1f}–{bins[i+1]:.1f})" for i in range(4)]
# ----------------------------------
# Prepare data (drop NaNs properly)
# ----------------------------------
anova_df = df[['binned_ADFSCI_hypo', 'dSBP']].dropna()

# Ensure ordered categories
anova_df['binned_ADFSCI_hypo'] = pd.Categorical(
    anova_df['binned_ADFSCI_hypo'],
    categories=[f'Q{i+1}' for i in range(4)],
    ordered=True
)

# ----------------------------------
# One-way ANOVA
# ----------------------------------
groups = [
    anova_df.loc[anova_df['binned_ADFSCI_hypo'] == q, 'dSBP'].values
    for q in anova_df['binned_ADFSCI_hypo'].cat.categories
]

anova_F, anova_p = stats.f_oneway(*groups)

print("One-way ANOVA for ΔSBP across OH domain score quartiles")
print(f"F = {anova_F:.3f}, p = {anova_p:.4f}")

tukey = pairwise_tukeyhsd(
    endog=anova_df['dSBP'],
    groups=anova_df['binned_ADFSCI_hypo'],
    alpha=0.05
)

print("\nPost-hoc Tukey HSD results:")
print(tukey.summary())



sns.boxplot(
    data=df,
    x='binned_ADFSCI_hypo',
    y='dSBP',
    color=MAIN_COLOR,
    showfliers=False,
    saturation=1,
    medianprops=dict(linewidth=3, color=MEDIAN_COLOR),
    boxprops=dict(linewidth=0, alpha=0.7),
    ax=ax[0, 1]
)

for y in [-60, -40, -20, 0, 20]:
    ax[0, 1].axhline(y=y, color='#b3b3b3', alpha=0.6, zorder=0)

ax[0, 1].set_xlabel('ADFSCI OH subscale quartiles', fontsize=14)
ax[0, 1].set_ylabel('ΔSBP (mmHg)', fontsize=14)
ax[0, 1].set_xticklabels(range_labels)
ax[0, 1].set_title('b', loc='left', fontsize=16, fontweight='bold')
ax[0, 1].spines[['left', 'bottom']].set_linewidth(0)
ax[0, 1].tick_params(width=0, length=0)

# =========================
# Fig 2c — ΔSBP vs OH score
# =========================
sns.regplot(
    data=df,
    x='OH_domain_score',
    y='dSBP',
    scatter_kws={'linewidths': 0},
    color=plot.exp_color1,
    ax=ax[1, 0]
)

sp, sp_p = spearmanr(df['dSBP'], df['OH_domain_score'])

ax[1, 0].set_xlabel('OH domain score', fontsize=14)
ax[1, 0].set_ylabel(r'$\Delta$SBP (mmHg)', fontsize=14)
ax[1, 0].set_title(
    f'c  Spearman ρ = {sp:.2f} (p={sp_p:.3f})',
    loc='center', fontsize=14
)



# =========================
# Fig 2d — ΔDBP vs OH score
# =========================
sns.regplot(
    data=df,
    x='OH_domain_score',
    y='dDBP',
    scatter_kws={'linewidths': 0},
    color=plot.exp_color1,
    ax=ax[1, 1]
)

sp, sp_p = spearmanr(df['dDBP'], df['OH_domain_score'])

ax[1, 1].set_xlabel('OH domain score', fontsize=14)
ax[1, 1].set_ylabel(r'$\Delta$DBP (mmHg)', fontsize=14)
ax[1, 1].set_title(
    f'd  Spearman ρ = {sp:.2f} (p={sp_p:.3f})\n',
    loc='center', fontsize=14
)

# =========================
# Finalize
# =========================
fig.tight_layout()
fig.savefig('Figure_2_combined.pdf', dpi=300, bbox_inches='tight')
fig.show()


In [ ]:
#Figure 3

# Clean setup
adfsci_cols = ['OH_domain_score', 'AD_domain_score', 'total_score']
names = ['OH score', 'AD score', 'total score']
metric = 'has_OH_SBP'
lr = LogisticRegression(class_weight='balanced')
cutoff_threshold = 0.4
results = {}

# Fit models and collect metrics
for name, col in zip(names, adfsci_cols): 
    model = lr.fit(df[col].to_numpy().reshape(-1, 1), df[metric].to_numpy())
    prob = model.predict_proba(df[col].to_numpy().reshape(-1, 1))[:, 1]
    pred = (prob >= cutoff_threshold).astype(int)

    fpr, tpr, _ = roc_curve(df[metric], prob)
    auc = roc_auc_score(df[metric], prob)
    acc = accuracy_score(df[metric], pred)
    cm = confusion_matrix(df[metric], pred)

    results[name] = {'fpr': fpr, 'tpr': tpr, 'auc': auc, 'acc': acc, 'cm': cm}

# Bootstrapping AUC for OH score
n_bootstraps = 1000
rng = np.random.RandomState(42)
bootstrapped_aucs = []


y_true = df[metric].to_numpy()
prediction_model = lr.fit(df['OH_domain_score'].to_numpy().reshape(-1, 1), y_true)
y_score = prediction_model.predict_proba(df['OH_domain_score'].to_numpy().reshape(-1, 1))[:, 1]
y_pred = (y_score >= cutoff_threshold).astype(int)
for _ in range(n_bootstraps):
    indices = rng.choice(np.arange(len(y_true)), size=len(y_true), replace=True)
    if len(np.unique(y_true[indices])) < 2:
        continue
    auc = roc_auc_score(y_true[indices], y_score[indices])
    bootstrapped_aucs.append(auc)

ci_lower = np.percentile(bootstrapped_aucs, 2.5)
ci_upper = np.percentile(bootstrapped_aucs, 97.5)
mean_auc = np.mean(bootstrapped_aucs)

# Classification correctness
correct_idx = np.where(y_pred == y_true)[0]
incorrect_idx = np.where(y_pred != y_true)[0]

# Plotting
fig, ax = plt.subplots(2, 2, figsize=(16, 16))

# ROC curve
for i, (name, res) in enumerate(results.items()):
    ax[0, 0].plot(res['fpr'], res['tpr'], label=f'{name} (AUC = {res["auc"]:.2f})',
                  linewidth=1.5, alpha=1 - 0.25 * i, color='crimson')
ax[0, 0].plot([0, 1], [0, 1], 'k--', alpha=0.7, linewidth=2.5)
ax[0, 0].set_xlim([-0.004, 1.0])
ax[0, 0].set_ylim([0.0, 1.01])
ax[0, 0].set_title('ROC Curve', size=16)
ax[0, 0].set_xlabel('False Positive Rate', size=16)
ax[0, 0].set_ylabel('True Positive Rate', size=16)
ax[0, 0].legend(loc='lower right', fontsize=16, frameon=False)
ax[0, 0].spines[['left', 'bottom']].set_linewidth(0)
ax[0, 0].tick_params(width=0, length=0, labelsize=16)




# Confusion matrix
sns.heatmap(results['OH score']['cm'], annot=True, fmt='d', cmap='Reds',
            xticklabels=['No OH', 'OH'], yticklabels=['No OH', 'OH'], ax=ax[0, 1])
ax[0, 1].set_title(f"Confusion Matrix \n Accuracy = {results['OH score']['acc']:.2f}", size=24)
ax[0, 1].tick_params(axis='both', labelsize=16)

tn, fp, fn, tp = results['OH score']['cm'].ravel()
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
ax[0, 1].set_title(f"Confusion Matrix \n Accuracy = {results['OH score']['acc']:.2f}\nSensitivity = {sensitivity:.2f}, Specificity = {specificity:.2f}", size=24)
ax[0, 1].tick_params(axis='both', labelsize=16)


# Classification scatter
ax[1, 0].scatter(df['OH_domain_score'].iloc[correct_idx],
                 y_score[correct_idx], c='crimson', s=80, edgecolors=None, linewidth=0,
                 label='Correct', marker='o', alpha=0.9)
ax[1, 0].scatter(df['OH_domain_score'].iloc[incorrect_idx],
                 y_score[incorrect_idx], c='black', s=80,
                 label='Misclassified', marker='x', alpha=0.9)
ax[1, 0].axhline(cutoff_threshold, color='gray', linestyle='--', linewidth=1.5, label=f'Threshold = {cutoff_threshold}')
ax[1, 0].set_title('Classification Result', fontsize=24)
ax[1, 0].set_xlabel('OH_domain_score', fontsize=24)
ax[1, 0].set_ylabel('Predicted Probability', fontsize=24)
ax[1, 0].set_ylim([0,1.0])
ax[1, 0].tick_params(axis='both', labelsize=16)
ax[1, 0].legend(fontsize=12, frameon=False)

# Histogram of bootstrapped AUCs
ax[1, 1].hist(bootstrapped_aucs, bins=30, color='lightcoral', alpha=0.7, linewidth=0)
ax[1, 1].axvline(ci_lower, color='crimson', linestyle='--', label=f'2.5% = {ci_lower:.2f}')
ax[1, 1].axvline(ci_upper, color='crimson', linestyle='--', label=f'97.5% = {ci_upper:.2f}')
ax[1, 1].axvline(mean_auc, color='crimson', linestyle='-', label=f'Mean = {mean_auc:.2f}', linewidth=3)
ax[1, 1].set_title('Bootstrapped Distribution of AUC (Logistic Regression)', size=18)
ax[1, 1].set_xlabel('AUC', size=24)
ax[1, 1].set_ylabel('Frequency', size=24)
ax[1, 1].legend(fontsize=14, frameon=False)
ax[1, 1].spines[['left', 'bottom']].set_linewidth(0)
ax[1, 1].tick_params(width=0, length=0, labelsize=16)

plt.tight_layout()
plt.savefig('Figure_3_ROC_analysis.pdf', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# Table 4. Discriminative performance
# Reuses variables computed in Figure 3: results, y_true

import pandas as pd

tn, fp, fn, tp = results['OH score']['cm'].ravel()

n_total     = len(y_true)
n_oh_pos    = int(y_true.sum())
n_oh_neg    = n_total - n_oh_pos

accuracy    = (tp + tn) / n_total
sensitivity = tp / (tp + fn)
specificity = tn / (tn + fp)
fnr         = fn / (tp + fn)
fpr_val     = fp / (tn + fp)
f1          = 2 * tp / (2 * tp + fp + fn)

table4_data = {
    'Property': [
        'Total participants',
        'Gold standard OH+',
        'Gold standard OH-',
        'True positive (OH+, predicted OH+)',
        'False negative (OH+, predicted OH-)',
        'False positive (OH-, predicted OH+)',
        'True negative (OH-, predicted OH-)',
        'Accuracy (TP + TN) / N',
        'Sensitivity TP / (TP + FN)',
        'Specificity TN / (TN + FP)',
        'False negative rate FN / (TP + FN)',
        'False positive rate FP / (TN + FP)',
        'F1 score 2TP / (2TP + FP + FN)',
    ],
    'Value': [
        n_total,
        n_oh_pos,
        n_oh_neg,
        int(tp),
        int(fn),
        int(fp),
        int(tn),
        f'{accuracy:.3f}',
        f'{sensitivity:.3f}',
        f'{specificity:.3f}',
        f'{fnr:.3f}',
        f'{fpr_val:.3f}',
        f'{f1:.3f}',
    ],
}

df_table4 = pd.DataFrame(table4_data)
print('Table 4. Discriminative performance')
print(df_table4.to_string(index=False))
df_table4

In [ ]:
## Figure 4

import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib as mpl


# Extract dSBP (SBP Drop) and ensure numeric
df['dSBP'] = pd.to_numeric(df['dSBP'], errors='coerce').fillna(0)

# Prepare linear regressions
X_freq = sm.add_constant(df['OH_frequency_score'])
X_sev = sm.add_constant(df['OH_severity_score'])
X_comb = sm.add_constant(df['OH_domain_score'])

model_freq = sm.OLS(df['dSBP'], X_freq).fit()
model_sev = sm.OLS(df['dSBP'], X_sev).fit()
model_comb = sm.OLS(df['dSBP'], X_comb).fit()

# Group comparisons
group0 = df[df['has_OH_SBP']==0]
group1 = df[df['has_OH_SBP']==1]

# print out the results
print("Linear Regression R²:")
print(f"  Frequency: {model_freq.rsquared:.4f}, p-value: {model_freq.pvalues['OH_frequency_score']:.4f}")
print(f"  Severity: {model_sev.rsquared:.4f}, p-value: {model_sev.pvalues['OH_severity_score']:.4f}")
print(f"  Combined: {model_comb.rsquared:.4f}, p-value: {model_comb.pvalues['OH_domain_score']:.4f}")

# Calculate p-values for boxplots using Mann-Whitney U test
mw_p_freq = mannwhitneyu(df[df['has_OH_SBP'] == 0]['OH_frequency_score'], 
                         df[df['has_OH_SBP'] == 1]['OH_frequency_score'],
                         alternative='less').pvalue
mw_p_sev = mannwhitneyu(df[df['has_OH_SBP'] == 0]['OH_severity_score'],
                        df[df['has_OH_SBP'] == 1]['OH_severity_score'],
                        alternative='less').pvalue
mw_p_comb = mannwhitneyu(df[df['has_OH_SBP'] == 0]['OH_domain_score'],
                         df[df['has_OH_SBP'] == 1]['OH_domain_score'],
                         alternative='less').pvalue

# R² values
r2_freq = model_freq.rsquared
p_freq = model_freq.pvalues['OH_frequency_score']

r2_sev = model_sev.rsquared
p_sev = model_sev.pvalues['OH_severity_score']

r2_comb = model_comb.rsquared
p_comb = model_comb.pvalues['OH_domain_score']


# Ensure relevant columns are numeric
df['has_OH_SBP'] = pd.to_numeric(df['has_OH_SBP'], errors='coerce')
df['OH_frequency_score'] = pd.to_numeric(df['OH_frequency_score'], errors='coerce')
df['OH_severity_score'] = pd.to_numeric(df['OH_severity_score'], errors='coerce')
df['OH_domain_score'] = pd.to_numeric(df['OH_domain_score'], errors='coerce')

# Melt the DataFrame for plotting
df_melted = pd.melt(
    df,
    id_vars=['has_OH_SBP'],
    value_vars=['OH_frequency_score', 'OH_severity_score', 'OH_domain_score'],
    var_name='Score_Type',
    value_name='Score'
)

# Mann–Whitney U tests
mw_results = {}
for score in ['OH_frequency_score', 'OH_severity_score', 'OH_domain_score']:
    group1 = df[df['has_OH_SBP'] == 0][score]
    group2 = df[df['has_OH_SBP'] == 1][score]
    mw_p = mannwhitneyu(group1, group2, alternative='less').pvalue
    mw_results[score] = mw_p

# Label update with p-values
label_dict = {
    'OH_frequency_score': f'Frequency (p={mw_results["OH_frequency_score"]:.3f})',
    'OH_severity_score': f'Severity (p={mw_results["OH_severity_score"]:.3f})',
    'OH_domain_score': f'Combined (p={mw_results["OH_domain_score"]:.3f})'
}

print("Mann-Whitney U test p-values for OH score types:")
for score_type, p_val in mw_results.items():
    print(f"  {score_type}: p = {p_val:.4f}")


df_melted['Score_Type'] = df_melted['Score_Type'].map(label_dict)

# =========================
# Global style
# =========================
sns.set_style("white")
MAIN_RED = "#f08080"
GRAY = "#c6c6c6"

# =========================
# Create canvas
# =========================
plot = EmptyPlot(nrows=1, ncols=3, figsize=(21, 6))
fig, ax = plot.fig, plot.ax

# ======================================================
# Panel 4a — OH score comparisons (Frequency / Severity / Combined)
# ======================================================
sns.boxplot(
    x='Score_Type',
    y='Score',
    hue='has_OH_SBP',
    data=df_melted,
    palette=[GRAY, MAIN_RED],
    linewidth=1.2,
    saturation=1,
    medianprops=dict(linewidth=3, color='darkred'),
    showfliers=False,
    ax=ax[0]
)

ax[0].set_xlabel('')
ax[0].set_ylabel('Score', fontsize=13)
ax[0].set_title('a', loc='left', fontsize=16, fontweight='bold')
ax[0].legend(title='OH Group', labels=['No OH', 'OH'], frameon=False)
sns.despine(ax=ax[0], top=True, right=True, left=True)

# ======================================================
# Panel 4b — Context-level Spearman correlations
# ======================================================
# Compute Spearman correlations
contexts = ['combined_transfer', 'combined_meal', 'combined_exercise']
corr_values = [-1*spearmanr(df[context], df['dSBP'], nan_policy='omit').correlation for context in contexts]
labels = ['Transfer', 'Meal', 'Exercise']

# Create a DataFrame for plotting
corr_df = pd.DataFrame({'Context': labels, 'Spearman_r': corr_values}).sort_values("Spearman_r", ascending=False)

# Normalize for color mapping
norm = mpl.colors.Normalize(vmin=corr_df["Spearman_r"].min(), vmax=corr_df["Spearman_r"].max())
cmap = sns.light_palette("red", as_cmap=True)

print(corr_df)

norm = mpl.colors.Normalize(
    vmin=corr_df["Spearman_r"].min(),
    vmax=corr_df["Spearman_r"].max()
)
cmap = sns.light_palette("red", as_cmap=True)

sns.barplot(
    data=corr_df,
    y="Context",
    x="Spearman_r",
    palette=cmap(norm(corr_df["Spearman_r"])),
    orient='h',
    ax=ax[1]
)

for i, v in enumerate(corr_df["Spearman_r"]):
    ax[1].text(
        v + 0.01 if v >= 0 else v - 0.01,
        i,
        f"{v:.2f}",
        va='center',
        ha='left' if v >= 0 else 'right',
        fontsize=10
    )

ax[1].axvline(0, color='black', linewidth=1.2)
ax[1].set_xlabel("Spearman ρ", fontsize=13)
# remove x ticks and marks
ax[1].set_xticks([])
# remove x tick marks

ax[1].set_ylabel('')

ax[1].set_title('b', loc='left', fontsize=16, fontweight='bold')
sns.despine(ax=ax[1], left=True, bottom=True)



# ======================================================
# Panel 4c — Symptom-level Spearman correlations
# ======================================================
# Define symptom row identifiers and corresponding labels
symptom_names = [
    'Dizziness', 'Light-headedness', 'Blurred-vision', 'Nausea',
    'Weakness', 'Confusion', 'Fatigue', 'Passing-out'
]
row_indices = range(1, 9)

# Frequency questions across contexts (transfers, meals, exercise)
freq_contexts = ['Q17', 'Q19', 'Q20', 'Q21']
freq_symptom_cols = {
    name: [f'{ctx}_R{r}_C1' for ctx in freq_contexts]
    for name, r in zip(symptom_names, row_indices)
}

# Severity questions across contexts (C1 = transfers, C2 = meals, C3 = exercise)
sev_contexts = ['C1', 'C2', 'C3']
sev_symptom_cols = {
    name: [f'Q22_R{r}_{c}' for c in sev_contexts]
    for name, r in zip(symptom_names, row_indices)
}

# Compute total frequency, severity, and combined scores for each symptom
symptom_scores = pd.DataFrame(index=df.index)

for symptom in symptom_names:
    freq_sum = df[freq_symptom_cols[symptom]].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1)
    sev_sum = df[sev_symptom_cols[symptom]].apply(pd.to_numeric, errors='coerce').fillna(0).sum(axis=1)
    symptom_scores[f'{symptom}_freq'] = freq_sum
    symptom_scores[f'{symptom}_sev'] = sev_sum
    symptom_scores[f'{symptom}_comb'] = freq_sum + sev_sum

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score


# Calculate Spearman correlation with dSBP for each symptom (combined scores)
correlations = {
    symptom: spearmanr(symptom_scores[f'{symptom}_comb'], df['dSBP']).correlation
    for symptom in symptom_names
}
# print out the p-values for each symptom
# sort it by absolute correlation value
p_values_dict = {}
for symptom in symptom_names:
    corr, p_value = spearmanr(symptom_scores[f'{symptom}_comb'], df['dSBP'])
    p_values_dict[symptom] = p_value

# sort by absolute correlation value
sorted_symptoms = sorted(symptom_names, key=lambda s: abs(correlations[s]), reverse=True)
print("\nSorted by absolute correlation value:")
for symptom in sorted_symptoms:
    print(f"Symptom: {symptom}, Spearman ρ: {correlations[symptom]:.4f}, p-value: {p_values_dict[symptom]:.4f}")

# Calculate AUC for each symptom predicting clinical OH
aucs = {}
for symptom in symptom_names:
    x = symptom_scores[f'{symptom}_comb'].fillna(0).to_numpy().reshape(-1, 1)
    y = df['has_OH_SBP']
    model = LogisticRegression(class_weight='balanced', max_iter=1000)
    model.fit(x, y)
    prob = model.predict_proba(x)[:, 1]
    aucs[symptom] = roc_auc_score(y, prob)

# Identify top symptom for each metric
best_corr_symptom = max(correlations, key=lambda k: abs(correlations[k]))
best_auc_symptom = max(aucs, key=aucs.get)

# Display results
{
    "Best Correlation with dSBP": {
        "Symptom": best_corr_symptom,
        "Spearman ρ": correlations[best_corr_symptom]
    },
    "Best AUC for Predicting Clinical OH": {
        "Symptom": best_auc_symptom,
        "AUC": aucs[best_auc_symptom]
    }
}

import matplotlib as mpl
# Prepare data
corr_values = [-1 * correlations[symptom] for symptom in symptom_names]
auc_values = [aucs[symptom] for symptom in symptom_names]
corr_df_symptom = pd.DataFrame({'Symptom': symptom_names, 'Spearman_r': corr_values})
corr_df_symptom = corr_df_symptom.sort_values("Spearman_r", ascending=False)



norm_sym = mpl.colors.Normalize(
    vmin=corr_df_symptom["Spearman_r"].min(),
    vmax=corr_df_symptom["Spearman_r"].max()
)

sns.barplot(
    data=corr_df_symptom,
    y="Symptom",
    x="Spearman_r",
    palette=cmap(norm_sym(corr_df_symptom["Spearman_r"])),
    orient='h',
    ax=ax[2]
)

for i, v in enumerate(corr_df_symptom["Spearman_r"]):
    ax[2].text(
        v + 0.01 if v >= 0 else v - 0.01,
        i,
        f"{v:.2f}",
        va='center',
        ha='left' if v >= 0 else 'right',
        fontsize=10
    )

ax[2].axvline(0, color='black', linewidth=1.2)
# remove x ticks and marks
ax[2].set_xticks([])
ax[2].set_xlabel("Spearman ρ", fontsize=13)
ax[2].set_ylabel('')
ax[2].set_title('c', loc='left', fontsize=16, fontweight='bold')
sns.despine(ax=ax[2], left=True, bottom=True)

# =========================
# Finalize
# =========================
fig.tight_layout()
fig.savefig("Figure_4_combined.pdf", dpi=300, bbox_inches='tight')
fig.show()
print("Figure 4")


In [ ]:
# Compute Spearman correlations and p-values for each context
context_corr_results = {
    context: spearmanr(df[context], df['dSBP'], nan_policy='omit')
    for context in ['combined_transfer', 'combined_meal', 'combined_exercise']
}

# Format results for display
context_corr_summary = {
    context: {
        'Spearman ρ': round(result.correlation, 3),
        'p-value': round(result.pvalue, 4)
    }
    for context, result in context_corr_results.items()
}

context_corr_summary

In [ ]:
### Supplementary Figure 1
print('OH severity results:')
results = known_groups_validity(df, 'OH_severity_score', group_column='has_OH_SBP')
print(f"p-value: {results['p_value']:.4f}, Cohen's d: {results['cohens_d']:.4f}")


print('OH frequency results:')
results = known_groups_validity(df, 'OH_frequency_score', group_column='has_OH_SBP')
print(f"p-value: {results['p_value']:.4f}, Cohen's d: {results['cohens_d']:.4f}")

# also do the spearman correlation

group1 = df[df['has_OH_SBP'] == 1]['OH_severity_score']
group2 = df[df['has_OH_SBP'] == 0]['OH_severity_score']

sp, sp_p = spearmanr(df['OH_severity_score'], df['dSBP'])
print(f"Spearman correlation between OH severity score and ΔSBP: ρ = {sp:.4f}, p-value = {sp_p:.4f}")
group1 = df[df['has_OH_SBP'] == 1]['OH_frequency_score']
group2 = df[df['has_OH_SBP'] == 0]['OH_frequency_score']

sp, sp_p = spearmanr(df['OH_frequency_score'], df['dSBP'])
print(f"Spearman correlation between OH frequency score and ΔSBP: ρ = {sp:.4f}, p-value = {sp_p:.4f}")

# =========================
# Create EmptyPlot canvas
# =========================
plot = EmptyPlot(nrows=1, ncols=2, figsize=(16, 8))
fig, ax = plot.fig, plot.ax

# =========================
# b — ΔSBP vs OH frequency score
# =========================
sns.regplot(
    data=df,
    x='OH_frequency_score',
    y='dSBP',
    scatter_kws={'linewidths': 0},
    color=plot.exp_color1,
    ax=ax[0]
)
sp, sp_p = spearmanr(df['dSBP'], df['OH_frequency_score'])
    
ax[0].set_xlabel('Symptom frequency score', fontsize=14)
ax[0].set_ylabel(r'$\Delta$SBP (mmHg)', fontsize=14)
ax[0].set_title(
    f'd  Spearman ρ = {sp:.2f} (p={sp_p:.3f})\n',
    loc='center', fontsize=14
)

# =========================
# a — ΔSBP vs OH severity score
# =========================
sns.regplot(
    data=df,
    x='OH_severity_score',
    y='dSBP',
    scatter_kws={'linewidths': 0},
    color=plot.exp_color1,
    ax=ax[1]
)

sp, sp_p = spearmanr(df['dSBP'], df['OH_severity_score'])

ax[1].set_xlabel('Symptom severity score', fontsize=14)
ax[1].set_ylabel(r'$\Delta$SBP (mmHg)', fontsize=14)
ax[1].set_title(
    f'c  Spearman ρ = {sp:.2f} (p={sp_p:.3f})',
    loc='center', fontsize=14
)


plt.tight_layout()
plt.savefig('Figure_R1C11_OH_severity_frequency_vs_dSBP.pdf', dpi=300, bbox_inches='tight')
plt.show()
print("Supplementary Figure 1")



In [ ]:
# data analysis for the global dataset
# import data
folderName = '../Data/'
fileName = 'global-dataset.xlsx'
xls = pd.ExcelFile(folderName+fileName)
df_global_raw = xls.parse(xls.sheet_names[0])

df_global_raw.head(5)
print(df_global_raw.shape)

In [ ]:
# Remove header row
df_cleaned = df_global_raw.iloc[1:].reset_index(drop=True)

# Build column category map
column_categories = {
    "Demographics": [col for col in df_global_raw.columns if re.match(r"Q(1[0-8]|[1-9])(_.*)?$", col)],
    "AD": [col for col in df_global_raw.columns if re.match(r"Q(3[7-8]|4[0-8])(_.*)?$", col)],
    "OH": [col for col in df_global_raw.columns if re.match(r"Q(4[9]|50|5[2-9])(_.*)?$", col)],
    "OHSA_symptoms": [col for col in df_global_raw.columns if re.match(r"Q61(_.*)?$", col)],
    "OHSA_activity": [col for col in df_global_raw.columns if re.match(r"Q62(_.*)?$", col)],
    "weekly_OH": [col for col in df_global_raw.columns if re.match(r"Q63(_.*)?$", col)],
    "Weekly_AD": [col for col in df_global_raw.columns if re.match(r"Q64(_.*)?$", col)],
}

# Flatten all selected columns
all_selected_cols = sum(column_categories.values(), [])

# Keep only those columns in df_raw
df_global = df_cleaned[all_selected_cols]

# calculate completeness for each category
for category, cols in column_categories.items():
    if cols:  # avoid division by zero
        df_global[f"{category}_completeness"] = df_global[cols].notna().sum(axis=1) / len(cols)
    else:
        df_global[f"{category}_completeness"] = 0.0

# Calculate total completeness for AD + OH
ad_oh_cols = column_categories["AD"] + column_categories["OH"]
if ad_oh_cols:
    df_global["total_completeness"] = df_global[ad_oh_cols].notna().sum(axis=1) / len(ad_oh_cols)
else:
    df_global["total_completeness"] = 0.0

In [ ]:
## apply filters after selecting specific variables

# only keep male/female participants in Q1
df_selected = df_global[df_global['Q1'].isin([1, 2])]

## filter out the incomplete participants
# Set completeness threshold
threshold = 0.9 # we used 90% for each domain in the manuscript

# Filter by score-specific completeness
df_oh_filtered = df_selected[df_selected['OH_completeness'] >= threshold].copy()

# Replace missing OH values with group median (grouped by Sex)
for col in column_categories['OH']:
    df_oh_filtered[col] = pd.to_numeric(df_oh_filtered[col], errors='coerce')
    df_oh_filtered[col] = df_oh_filtered.groupby('Q1')[col].transform(
        lambda x: x.fillna(x.median())
    )

# Recalculate scores after filling
df_oh_filtered['OH_domain_score'] = df_oh_filtered[column_categories['OH']].sum(axis=1)

In [ ]:
## Characteristics Table for global dataset
def calculate_injury_months(date_series):
    def parse_injury_date(date_val):        
        date_str = str(date_val).strip()
        # Try MM-YYYY format first
        try:
            return pd.to_datetime(date_str, format='%m-%Y')
        except:
            pass
        
        # Try YYYY format
        try:
            return pd.to_datetime(date_str, format='%Y')
        except:
            pass
        
        # Try common datetime formats
        try:
            return pd.to_datetime(date_str)
        except:
            return pd.NaT

    # Parse the dates
    parsed_dates = date_series.apply(parse_injury_date)
    ref = pd.Period('2026-01', freq='M')
    injury_periods = parsed_dates.dt.to_period('M')
    injury_months = (ref - injury_periods).map(lambda x: x.n if pd.notna(x) else pd.NA)
    
    return injury_months

df_oh_filtered['injury_months'] = calculate_injury_months(df_oh_filtered['Q14_DATE'])
df_oh_filtered['Q3'] = pd.to_numeric(df_oh_filtered['Q3'], errors='coerce')  # Age
df_oh_filtered['OH_reported'] = df_oh_filtered['Q49'].map({1: 'Yes', 2: 'No', 3: 'Unsure', 4: 'Dont know OH'})
# Calculate time since injury in months
# for Q15, map 1: C1-4, 2: C5-8, 3: T1-4, 4: T5-18, 5: T9-12, 6: L1-5, 7: Other
df_oh_filtered['NLI'] = df_oh_filtered['Q15'].map({
    1: 'C1-4',
    2: 'C5-8',
    3: 'T1-4',
    4: 'T5-8',
    5: 'T9-12',
    6: 'L1-5',
    7: 'Other'
})
# for Q17, map 1: A, 2: B, 3: C, 4: D, 5: Other
df_oh_filtered['AIS_Grade'] = df_oh_filtered['Q17'].map({
    1: 'A',
    2: 'B',
    3: 'C',
    4: 'D',
    5: 'Other'
})

table5 = {
    "Total questionnaire collected": len(df_global),
    "Completeness Threshold": threshold,
    "Sample Size (n)": len(df_oh_filtered),
    "Mean ADFSCI OH Domain Score": f"{round(df_oh_filtered['OH_domain_score'].mean(), 1)} ± {round(df_oh_filtered['OH_domain_score'].std(), 1)}",
    "Range of ADFSCI OH Domain Score": f"{round(df_oh_filtered['OH_domain_score'].min(), 1)} – {round(df_oh_filtered['OH_domain_score'].max(), 1)}",
    "OH Reported": f"{df_oh_filtered['OH_reported'].value_counts().get('Yes', 0)} / {len(df_oh_filtered)}",
    "Age (years)": f"{round(df_oh_filtered['Q3'].mean(), 1)} ± {round(df_oh_filtered['Q3'].std(), 1)}",
    "Sex (M/F)": f"{(df_oh_filtered['Q1'] == 1).sum()}/{(df_oh_filtered['Q1'] == 2).sum()}",
    "Injury Level": f"C:{(df_oh_filtered['NLI'].str[0] == 'C').sum()}/T:{(df_oh_filtered['NLI'].str[0] == 'T').sum()}/L:{(df_oh_filtered['NLI'].str[0] == 'L').sum()}/Other:{(df_oh_filtered['NLI'].str[0] == 'O').sum()}",
    "AIS Grade": ', '.join([f"{grade}: {(df_oh_filtered['AIS_Grade'] == grade).sum()}" for grade in sorted(df_oh_filtered['AIS_Grade'].unique())]),
    "Time Since Injury (months)": f"{round(df_oh_filtered['injury_months'].mean()/12, 1)} ± {round(df_oh_filtered['injury_months'].std() / 12, 1)}"
}

# Convert dictionary to DataFrame
table5_df = pd.DataFrame(list(table5.items()), columns=["Characteristic", "Value"])

display(table5_df)

In [ ]:
# Supplementary Figure 2
# correlations between PROM scores with self-reported OH, resting BP, etc.

df_oh_filtered_new = df_oh_filtered[df_oh_filtered['OH_reported'].isin(['Yes', 'No'])]
df_oh_filtered_new['OH_reported'] = df_oh_filtered_new['OH_reported'].map({'Yes': 1, 'No': 0})
results = known_groups_validity(df_oh_filtered_new, 'OH_domain_score', group_column='OH_reported')
print(f"Known-groups validity (self-reported OH) - p-value: {results['p_value']:.4f}, Cohen's d: {results['cohens_d']:.4f}")

plot = EmptyPlot(nrows=1, ncols=2, figsize=(16, 8))
fig, ax = plot.fig, plot.ax
sns.boxplot(
    data=df_oh_filtered_new,
    x='OH_reported',
    y='OH_domain_score',
    color=MAIN_COLOR,
    saturation=1,
    medianprops=dict(linewidth=3, color=MEDIAN_COLOR),
    boxprops=dict(linewidth=0, alpha=0.7),
    showfliers=False,
    ax=ax[0]
)




ax[0].set_xticklabels(['OH-', 'OH+'])
ax[0].set_xlabel('Self-reported OH', fontsize=14)


# convergent validity: correlations between PROM scores and resting BP
# SBP parsing
df_oh_filtered['Q63'] = df_oh_filtered['Q63'].astype(str)
df_oh_filtered[['SBP', 'DBP']] = df_oh_filtered['Q63'].str.extract(r'(\d+)\s*/\s*(\d+)').astype(float)


df_oh_filtered['SBP_Group'] = df_oh_filtered['SBP'].apply(lambda sbp: '<100 mmHg' if sbp < 100 else 'Other')
# only use the non nan values for the correlation analysis
df_oh_filtered_nonan = df_oh_filtered.dropna(subset=['OH_domain_score', 'SBP'])
# also remove SBP < 20 because it's physiologically implausible and likely a data entry error
df_oh_filtered_nonan = df_oh_filtered_nonan[df_oh_filtered_nonan['SBP'] >= 20]


# print out the number of non-nan samples
print(f"Number of non-nan samples for OH_domain_score and SBP: {len(df_oh_filtered_nonan)}")
sp, sp_p = spearmanr(df_oh_filtered_nonan['OH_domain_score'], df_oh_filtered_nonan['SBP'])
print(f"Convergent validity (resting SBP) - Spearman ρ: {sp:.4f}, p-value: {sp_p:.4f}")

sns.regplot(
    data=df_oh_filtered_nonan,
    x='SBP',
    y='OH_domain_score',
    scatter_kws={'linewidths': 0},
    color=plot.exp_color1,
    ax=ax[1]
)

# xlim and ylim

ax[1].set_xlabel('Resting SBP (mmHg)', fontsize=14)
ax[1].set_ylabel('ADFSCI OH Domain Score', fontsize=14)
ax[1].set_title(
    f'Convergent Validity - Spearman ρ = {sp:.2f} (p={sp_p:.3f})',
    loc='center', fontsize=14
)
fig.tight_layout()
fig.savefig('Figure_S2_OH_domain_score_convergent_validity.pdf', dpi=300, bbox_inches='tight')
fig.show()
print("Supplementary Figure 2")



In [ ]:
## Table 6: Unadjusted OR table (SBP cutoff: <110 mmHg)

import statsmodels.api as sm

# Column selection matching 20250606 (specific question IDs, no Q1 filter)
_demographics_cols = [col for col in df_global_raw.columns if re.match(r'Q(1|3|6|14|15|16|17)(_.*)?$', col)]
_medications_cols  = [col for col in df_global_raw.columns if re.match(r'Q(20|21|22)(_[^ ]+)?$', col)]
_knowledge_cols    = [col for col in df_global_raw.columns if re.match(r'Q(2[3-9]|3[0-6])(_.*)?$', col)]
_ad_cols  = [col for col in df_global_raw.columns if re.match(r'Q(3[7-8]|4[0-8])(_.*)?$', col)]
_oh_cols  = [col for col in df_global_raw.columns if re.match(r'Q(4[9]|50|5[2-9])(_.*)?$', col)]
_bp_cols  = [col for col in df_global_raw.columns if re.match(r'Q(62|63|65)(_.*)?$', col)]

_all_cols = _demographics_cols + _medications_cols + _knowledge_cols + _ad_cols + _oh_cols + _bp_cols
_df_selected = df_global_raw[_all_cols]

# Completeness (AD and OH only)
_completeness = pd.DataFrame(index=df_global_raw.index)
_all_q = []
for _sec, _cols in [('AD', _ad_cols), ('OH', _oh_cols)]:
    _all_q.extend(_cols)
    _completeness[_sec + '_completeness'] = df_global_raw[_cols].notnull().sum(axis=1) / len(_cols)
_all_q = list(set(_all_q))
_completeness['total_completeness'] = df_global_raw[_all_q].notnull().sum(axis=1) / len(_all_q)

_df_combined = pd.concat([_df_selected, _completeness], axis=1)
_df_combined = _df_combined.iloc[1:].reset_index(drop=True)  # drop header row

_df_oh = _df_combined[_df_combined['OH_completeness'] >= threshold].copy()
for col in _oh_cols:
    _df_oh[col] = _df_oh.groupby('Q1')[col].transform(lambda x: x.fillna(x.median()))
_df_oh['OH_Score'] = _df_oh[_oh_cols].sum(axis=1)

# Binary outcome (top 25%)
_t75 = _df_oh['OH_Score'].quantile(0.75)
_df_oh['High_OH_Burden'] = (_df_oh['OH_Score'] >= _t75).astype(int)

# Predictors
_df_oh['Sex']          = _df_oh['Q1'].map({1: 'Male', 2: 'Female'})
_df_oh['Age_Group']    = pd.cut(pd.to_numeric(_df_oh['Q3'], errors='coerce'), bins=[0, 29, 150], labels=['<30', '30+'])
_df_oh['Injury_Level'] = _df_oh['Q15'].apply(lambda x: 'High' if x in [1, 2, 3] else 'Low')
_df_oh['OH_Reported']  = _df_oh['Q49'].map({1: 'Yes', 2: 'No', 3: 'No'})
_df_oh['Q63']          = _df_oh['Q63'].astype(str)
_df_oh[['SBP', 'DBP']] = _df_oh['Q63'].str.extract(r'(\d+)\s*/\s*(\d+)').astype(float)
_df_oh['SBP_Group']    = _df_oh['SBP'].apply(lambda sbp: '<110 mmHg' if sbp < 110 else 'Other')

predictors = ['Sex', 'Age_Group', 'Injury_Level', 'OH_Reported', 'SBP_Group']
ref_map = {'Sex': 'Male', 'Age_Group': '<30', 'Injury_Level': 'Low',
           'OH_Reported': 'No', 'SBP_Group': 'Other'}

results = []
for var in predictors:
    data = _df_oh[[var, 'High_OH_Burden']].dropna().copy()
    dummies = pd.get_dummies(data[var])
    target = [c for c in dummies.columns if c != ref_map[var]][0]
    data['Predictor'] = dummies[target]
    X = sm.add_constant(data['Predictor']).astype(float)
    y = data['High_OH_Burden'].astype(int)
    if X['Predictor'].nunique() < 2:
        continue
    model = sm.Logit(y, X).fit(disp=False)
    coef, se, pval = model.params['Predictor'], model.bse['Predictor'], model.pvalues['Predictor']
    OR, CIl, CIh = np.exp(coef), np.exp(coef - 1.96*se), np.exp(coef + 1.96*se)
    results.append({
        'Predictor': f'{target} (vs. {ref_map[var]})',
        'OR': round(OR, 2),
        '95% CI': f'{round(CIl, 2)} ' + chr(0x2013) + f' {round(CIh, 2)}',
        'p-value': '<0.001' if pval < 0.001 else round(pval, 3)
    })

summary_df_or110 = pd.DataFrame(results)
print(f'n = {len(_df_oh)} (OH completeness >= 90%)')
display(summary_df_or110)
